In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import lightgbm as lgb
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
import shap
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.regression.mixed_linear_model import MixedLM
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
import time

In [ ]:
ds = xr.open_dataset("df_final.nc")
df = ds.to_dataframe().reset_index()
df.head()

#features
features = [
    "temperature_2m_C_y", "wind_speed_10m_ms", "surface_pressure_hPa_y",
    "total_precipitation_mm", "total_column_ozone_y", "surface_solar_radiation_downward_Wm2",
    "sea_salt_aerosol_1", "organic_matter_aerosol_2", "nitric_oxide", "methane",
    "specific_humidity_kgkg", "boundary_layer_height_m", "mean_sea_level_pressure_hPa",
    "wind_u_component_10m_ms", "wind_v_component_10m_ms", "low_cloud_cover_percent",
    "evaporation_mm", "ethane", "formaldehyde", "potential_vorticity_Km2s",
    "relative_humidity_percent", "vertical_velocity_Pas"
]
target = "PCA_AQI"

# Train-Test Split
train_df = df[df["timestamp"] < "2023-01-01"].copy()
test_df = df[df["timestamp"] >= "2023-01-01"].copy()

#  Standardizasyon
scaler = StandardScaler()
train_df[features + [target]] = scaler.fit_transform(train_df[features + [target]])
test_df[features + [target]] = scaler.transform(test_df[features + [target]])


for df_ in [train_df, test_df]:
    df_["date"] = pd.to_datetime(df_["timestamp"]).dt.to_period("M")
    df_["year"] = df_["date"].dt.year

train_df["year_scaled"] = train_df["year"] - train_df["year"].mean()
test_df["year_scaled"] = test_df["year"] - train_df["year"].mean()  # Test set için de aynı dönüşüm

features += ["year_scaled"]

# Mixed-Effect Model: Random Intercept + Random Slope (year)
formula = "PCA_AQI ~ " + " + ".join(features)
md = mixedlm(formula, train_df, groups=train_df["Country"])  
mdf = md.fit(method='lbfgs')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================================
# 1. MIXED-EFFECTS MODEL COMPONENT
# ============================================================================

class MixedEffectsExtractor:
    """Mixed-effects model ile country-specific random intercepts çıkarır"""
    
    def __init__(self):
        self.model = None
        self.random_effects = {}
        self.countries = None
        
    def fit(self, data, features, target, group_col):
        """Mixed-effects model fit et"""
        print("Fitting Mixed-Effects model...")
        
     
        df_me = data[features + [target, group_col]].copy()
        
        # Mixed-effects model formula
        formula = f"{target} ~ " + " + ".join(features)
        
        # Model fit 
        self.model = MixedLM.from_formula(
            formula, 
            data=df_me, 
            groups=df_me[group_col],
            re_formula="1"  # Random intercept
        )
        
        try:
            result = self.model.fit(reml=True, method='lbfgs')
            print("Mixed-effects model fitted successfully!")
            
            # extract Random effects 
            self.countries = df_me[group_col].unique()
            random_effects_df = result.random_effects
            
            for country in self.countries:
                if country in random_effects_df.index:
                    self.random_effects[country] = random_effects_df.loc[country]['Group']
                else:
                    self.random_effects[country] = 0.0
                    
            print(f"Extracted random effects for {len(self.random_effects)} countries")
            
        except Exception as e:
            print(f"Mixed-effects model fitting failed: {e}")
            # Fallback
            self.countries = df_me[group_col].unique()
            for country in self.countries:
                self.random_effects[country] = 0.0
                
        return self
    
    def get_random_effects(self, countries):
        """Verilen ülkeler için random effects döndür"""
        effects = []
        for country in countries:
            effects.append(self.random_effects.get(country, 0.0))
        return np.array(effects)

# ============================================================================
# 2. GAUSSIAN PROCESS COMPONENT  
# ============================================================================

class SpatialGaussianProcess:
    """Spatial dependencies için Gaussian Process"""
    
    def __init__(self, length_scale=1.0):
        kernel = C(1.0) * RBF(length_scale=length_scale)
        self.gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6)
        self.fitted = False
        
    def fit(self, coordinates, values):
        """GP'yi fit et"""
        try:
            self.gp.fit(coordinates, values)
            self.fitted = True
            print("Gaussian Process fitted successfully!")
        except Exception as e:
            print(f"GP fitting failed: {e}")
            self.fitted = False
        return self
    
    def predict(self, coordinates):
        """Spatial features predict et"""
        if not self.fitted:
            return np.zeros(len(coordinates))
        
        try:
            predictions, _ = self.gp.predict(coordinates, return_std=True)
            return predictions
        except:
            return np.zeros(len(coordinates))

# ============================================================================
# 3. LSTM ENCODER COMPONENT
# ============================================================================

class LSTMEncoder(nn.Module):
    """Temporal dependencies için LSTM encoder"""
    
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super(LSTMEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, 
            num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size = x.size(0)
        
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        
        # LSTM forward pass
        out, (hidden, cell) = self.lstm(x, (h0, c0))
        
        # Son time step'in output'unu al
        last_output = out[:, -1, :]  # (batch_size, hidden_dim)
        
        return self.dropout(last_output)

# ============================================================================
# 4. DIFFUSION MODEL COMPONENT
# ============================================================================

class DiffusionModel(nn.Module):
    """Denoising Diffusion Probabilistic Model"""
    
    def __init__(self, input_dim, hidden_dim=128, timesteps=300):
        super(DiffusionModel, self).__init__()
        self.timesteps = timesteps
        
        # Noise schedule
        self.register_buffer('betas', self._linear_schedule(timesteps))
        self.register_buffer('alphas', 1.0 - self.betas)
        self.register_buffer('alphas_cumprod', torch.cumprod(self.alphas, dim=0))
        
        # Time embedding
        self.time_embedding = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        
        # Denoising network
        self.denoiser = nn.Sequential(
            nn.Linear(input_dim + 32, hidden_dim),  # +32 for time embedding
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, input_dim)
        )
        
    def _linear_schedule(self, timesteps, beta_start=1e-4, beta_end=2e-2):
        """Linear noise schedule"""
        return torch.linspace(beta_start, beta_end, timesteps)
    
    def forward_diffusion(self, x0, t):
        """Add noise to clean data"""
        noise = torch.randn_like(x0)
        sqrt_alphas_cumprod_t = torch.sqrt(self.alphas_cumprod[t])
        sqrt_one_minus_alphas_cumprod_t = torch.sqrt(1.0 - self.alphas_cumprod[t])
        
        # Reshape for broadcasting
        sqrt_alphas_cumprod_t = sqrt_alphas_cumprod_t.view(-1, 1)
        sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod_t.view(-1, 1)
        
        noisy_x = sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise
        return noisy_x, noise
    
    def reverse_diffusion(self, xt, t):
        """Predict noise from noisy data"""
        # Time embedding
        t_emb = self.time_embedding(t.float().unsqueeze(-1))
        
        # Concatenate input with time embedding
        xt_with_time = torch.cat([xt, t_emb], dim=-1)
        
        # Predict noise
        predicted_noise = self.denoiser(xt_with_time)
        return predicted_noise

# ============================================================================
# 5. COMPLETE H-MED MODEL
# ============================================================================

class HMEDModel(nn.Module):
    """Hybrid Mixed-Effect Diffusion Model"""
    
    def __init__(self, feature_dim, hidden_dim=64, latent_dim=32, timesteps=300):
        super(HMEDModel, self).__init__()
        
        self.feature_dim = feature_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.timesteps = timesteps
        
        # Components
        self.lstm_encoder = LSTMEncoder(feature_dim, hidden_dim)
        self.diffusion = DiffusionModel(latent_dim, hidden_dim, timesteps)
        
        # Fusion layer (LSTM + GP + Random Effects → Latent)
        fusion_input_dim = hidden_dim + 1 + 1  # LSTM + GP + Random Effect
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        # Decoder (Latent → AQI)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, features, spatial_features, random_effects, training=True):
        """
        Args:
            features: (batch_size, seq_len, feature_dim) - temporal features
            spatial_features: (batch_size, 1) - GP spatial features
            random_effects: (batch_size, 1) - country random effects
            training: bool - training mode flag
        """
        batch_size = features.size(0)
        
        # 1. LSTM encoding for temporal features
        lstm_out = self.lstm_encoder(features)  # (batch_size, hidden_dim)
        
        # 2. Fuse all components
        fused_input = torch.cat([
            lstm_out,
            spatial_features.view(batch_size, -1),
            random_effects.view(batch_size, -1)
        ], dim=-1)
        
        latent = self.fusion(fused_input)  # (batch_size, latent_dim)
        
        if training:
            # 3. Diffusion process during training
            t = torch.randint(0, self.timesteps, (batch_size,)).to(features.device)
            noisy_latent, noise = self.diffusion.forward_diffusion(latent, t)
            predicted_noise = self.diffusion.reverse_diffusion(noisy_latent, t)
            
            # 4. Decode to final prediction
            prediction = self.decoder(latent)
            
            return prediction, noise, predicted_noise
        else:
            # Inference: direct decoding
            prediction = self.decoder(latent)
            return prediction

# ============================================================================
# 6. DATASET CLASS
# ============================================================================

class AirQualityDataset(Dataset):
    """Air Quality Dataset for H-MED"""
    
    def __init__(self, data, features, target, group_col, 
                 mixed_effects_extractor, spatial_gp, 
                 sequence_length=10):
        
        self.data = data.copy()
        self.features = features
        self.target = target
        self.group_col = group_col
        self.sequence_length = sequence_length
        self.me_extractor = mixed_effects_extractor
        self.spatial_gp = spatial_gp
        
        # Prepare sequences
        self._prepare_sequences()
        
    def _prepare_sequences(self):
        """Temporal sequences hazırla"""
        self.sequences = []
        
        # Her ülke için ayrı ayrı sequence oluştur
        for country in self.data[self.group_col].unique():
            country_data = self.data[self.data[self.group_col] == country].copy()
            country_data = country_data.sort_values('timestamp')
            
            # Random effect al
            random_effect = self.me_extractor.get_random_effects([country])[0]
            
            # Spatial feature al (lat, lon varsa)
            if 'lat' in country_data.columns and 'lon' in country_data.columns:
                coords = country_data[['lat', 'lon']].values[0:1]
                spatial_feature = self.spatial_gp.predict(coords)[0]
            else:
                spatial_feature = 0.0
            
            # Sequences oluştur
            for i in range(len(country_data) - self.sequence_length + 1):
                seq_data = country_data.iloc[i:i+self.sequence_length]
                
                # Features sequence
                feature_seq = seq_data[self.features].values
                
                # Target (son time step)
                target_val = seq_data[self.target].iloc[-1]
                
                self.sequences.append({
                    'features': feature_seq,
                    'target': target_val,
                    'spatial': spatial_feature,
                    'random_effect': random_effect,
                    'country': country
                })
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        item = self.sequences[idx]
        
        return {
            'features': torch.FloatTensor(item['features']),
            'target': torch.FloatTensor([item['target']]),
            'spatial': torch.FloatTensor([item['spatial']]),
            'random_effect': torch.FloatTensor([item['random_effect']])
        }

# ============================================================================
# 7. TRAINING FUNCTION
# ============================================================================

def train_hmed(model, train_loader, val_loader, num_epochs=50, learning_rate=3e-4):
    """H-MED model training"""
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    model.train()
    train_losses = []
    val_losses = []
    
    print("Starting H-MED training...")
    
    for epoch in range(num_epochs):
        # Training
        epoch_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            features = batch['features'].to(device)
            targets = batch['target'].to(device)
            spatial = batch['spatial'].to(device)
            random_effects = batch['random_effect'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            predictions, noise, predicted_noise = model(
                features, spatial, random_effects, training=True
            )
            
            # Loss: reconstruction + diffusion
            recon_loss = F.mse_loss(predictions, targets)
            diffusion_loss = F.mse_loss(predicted_noise, noise)
            
            # Combined loss
            total_loss = recon_loss + 0.1 * diffusion_loss
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_train_loss += total_loss.item()
        
        # Validation
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                features = batch['features'].to(device)
                targets = batch['target'].to(device)
                spatial = batch['spatial'].to(device)
                random_effects = batch['random_effect'].to(device)
                
                predictions = model(features, spatial, random_effects, training=False)
                val_loss = F.mse_loss(predictions, targets)
                epoch_val_loss += val_loss.item()
        
        model.train()
        
        # Losses
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        scheduler.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
            print(f"Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
            print("-" * 50)
    
    return train_losses, val_losses

# ============================================================================
# 8. EVALUATION FUNCTION
# ============================================================================

def evaluate_hmed(model, test_loader, scaler):
    """H-MED model evaluation"""
    
    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features'].to(device)
            batch_targets = batch['target'].to(device)
            spatial = batch['spatial'].to(device)
            random_effects = batch['random_effect'].to(device)
            
            batch_predictions = model(features, spatial, random_effects, training=False)
            
            predictions.extend(batch_predictions.cpu().numpy())
            targets.extend(batch_targets.cpu().numpy())
    
    predictions = np.array(predictions).flatten()
    targets = np.array(targets).flatten()
    
    # Inverse transform if needed
    if scaler is not None:
        # Assuming target is the last column in the scaler
        dummy_features = np.zeros((len(predictions), scaler.n_features_in_))
        dummy_features[:, -1] = predictions
        predictions_orig = scaler.inverse_transform(dummy_features)[:, -1]
        
        dummy_features[:, -1] = targets
        targets_orig = scaler.inverse_transform(dummy_features)[:, -1]
    else:
        predictions_orig = predictions
        targets_orig = targets
    
    # Metrics
    mae = mean_absolute_error(targets_orig, predictions_orig)
    rmse = np.sqrt(mean_squared_error(targets_orig, predictions_orig))
    r2 = r2_score(targets_orig, predictions_orig)
    
    print("H-MED Model Performance:")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'predictions': predictions_orig,
        'targets': targets_orig
    }

# ============================================================================
# 9. MAIN IMPLEMENTATION
# ============================================================================

def run_hmed_pipeline(train_df, test_df, features, target, 
                      sequence_length=10, num_epochs=50):
    """Complete H-MED pipeline"""
    
    print("="*80)
    print("H-MED (Hybrid Mixed-Effect Diffusion) Model Pipeline")
    print("="*80)
    
    start_time = time.time()
    
    # 1. Mixed-Effects Model
    print("\n1. Extracting Mixed-Effects Random Intercepts...")
    me_extractor = MixedEffectsExtractor()
    me_extractor.fit(train_df, features, target, 'Country')
    
    # 2. Spatial Gaussian Process (eğer koordinat varsa)
    print("\n2. Fitting Spatial Gaussian Process...")
    if 'lat' in train_df.columns and 'lon' in train_df.columns:
        spatial_gp = SpatialGaussianProcess(length_scale=5.0)
        
        # Her ülke için ortalama koordinat al
        country_coords = train_df.groupby('Country')[['lat', 'lon']].mean()
        country_targets = train_df.groupby('Country')[target].mean()
        
        spatial_gp.fit(country_coords.values, country_targets.values)
    else:
        print("No spatial coordinates found, using dummy spatial features")
        spatial_gp = SpatialGaussianProcess()
        spatial_gp.fitted = False
    
    # 3. Datasets
    print("\n3. Preparing Datasets...")
    train_dataset = AirQualityDataset(
        train_df, features, target, 'Country',
        me_extractor, spatial_gp, sequence_length
    )
    
    test_dataset = AirQualityDataset(
        test_df, features, target, 'Country',
        me_extractor, spatial_gp, sequence_length
    )
    
    # 4. Data Loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
    val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
    
    # 5. Model
    print("\n4. Initializing H-MED Model...")
    model = HMEDModel(
        feature_dim=len(features),
        hidden_dim=64,
        latent_dim=32,
        timesteps=300
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # 6. Training
    print("\n5. Training H-MED Model...")
    train_losses, val_losses = train_hmed(
        model, train_loader, val_loader, 
        num_epochs=num_epochs, learning_rate=3e-4
    )
    
    # 7. Evaluation
    print("\n6. Evaluating H-MED Model...")
    results = evaluate_hmed(model, val_loader, scaler=None)
    
    # 8. Training time
    total_time = time.time() - start_time
    print(f"\nTotal training time: {total_time:.2f} seconds")
    
    # 9. Plot training curves
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('H-MED Training Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.scatter(results['targets'], results['predictions'], alpha=0.5)
    plt.plot([results['targets'].min(), results['targets'].max()], 
             [results['targets'].min(), results['targets'].max()], 'r--')
    plt.title(f"H-MED Predictions vs Targets\nR² = {results['r2']:.4f}")
    plt.xlabel('True Values')
    plt.ylabel('Predictions')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'model': model,
        'results': results,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'me_extractor': me_extractor,
        'spatial_gp': spatial_gp,
        'total_time': total_time
    }

# ============================================================================
# 10. RUN THE MODEL
# ============================================================================

if __name__ == "__main__":
    hmed_results = run_hmed_pipeline(
        train_df=train_df,
        test_df=test_df, 
        features=features,
        target=target,
        sequence_length=10,
        num_epochs=100)
    
    print("\n" + "="*80)
    print("H-MED MODEL RESULTS SUMMARY")
    print("="*80)
    print(f"MAE: {hmed_results['results']['mae']:.4f}")
    print(f"RMSE: {hmed_results['results']['rmse']:.4f}") 
    print(f"R²: {hmed_results['results']['r2']:.4f}")
    print(f"Training Time: {hmed_results['total_time']:.2f} seconds")
    print("="*80)